[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/19_evaluation.ipynb)

# Evaluation — did the fine-tune actually work?

> **Fine-tuning series — 19 of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. This notebook is self-contained: run **Setup**, then the rest.

## Setup (run first)

Self-contained: imports, model id, device flags, and a tiny inline dataset so this notebook runs on its own.

In [ ]:
# Environment check — record these versions with any results you report.
import importlib
for lib in ["torch", "transformers", "peft", "trl", "datasets",
            "bitsandbytes", "accelerate", "unsloth", "adapters"]:
    try:
        m = importlib.import_module(lib)
        print(f"{lib:14s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{lib:14s} not installed ({type(e).__name__})")

In [ ]:
# !pip install -U transformers peft trl bitsandbytes datasets accelerate
# !pip install unsloth   # only for the Unsloth section (CUDA only)
import gc, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # small + fast; swap for any causal LM
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16_OK = torch.cuda.is_available() and not BF16_OK   # fp16 needs a GPU
print("device:", device, "| bf16:", BF16_OK)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# (a) Instruction data -> for SFT / LoRA / QLoRA / Unsloth / instruction / prompt tuning
instructions = [
    {"instruction": "Give the capital of France.", "output": "Paris."},
    {"instruction": "What is 7 times 8?", "output": "56."},
    {"instruction": "Translate 'good morning' to Spanish.", "output": "Buenos d\u00edas."},
    {"instruction": "Name the largest planet.", "output": "Jupiter."},
    {"instruction": "Define photosynthesis in one line.",
     "output": "Plants converting sunlight, water, and CO2 into energy and oxygen."},
] * 8   # repeat just to have enough steps in the demo

# (b) Preference data -> for DPO (prompt + chosen/rejected, no reward model)
preferences = [
    {"prompt": "Explain gravity to a child.",
     "chosen": "Gravity is the pull that brings things down to the ground.",
     "rejected": "Gravity is a tensor field obeying the Einstein equations."},
    {"prompt": "Recommend a book.",
     "chosen": "Try 'The Hobbit' \u2014 a fun, easy adventure.",
     "rejected": "Read whatever, I don't care."},
] * 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

sft_ds = Dataset.from_list([to_chat_text(e) for e in instructions])
pref_ds = Dataset.from_list(preferences)
print(sft_ds[0]["text"][:160])

In [ ]:
# Shared trainer imports + a default LoRA config reused by several sections.
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])

## Evaluation — close the loop

Every other notebook *trains*. None of them answer the question that actually matters:
**did it get better?** Training loss going down is necessary, not sufficient — a model
can memorize the training set and still be useless. You measure three different things:

- **Intrinsic** — loss / **perplexity** on a *held-out* set (cheap, automatic).
- **Task metric** — accuracy / exact-match / F1 on examples with known answers.
- **Judgment** — human or **LLM-as-judge** ratings for open-ended quality.

Below we train a quick adapter, then score the **base vs. tuned** model every way.
(Demo numbers are tiny — they show the *mechanics*, not benchmark results.)

In [ ]:
# Train a quick LoRA adapter so we have a fine-tuned model. We keep just ONE model
# in memory: a PeftModel whose adapter we switch OFF to recover the *base* behavior,
# so the base-vs-tuned comparison never holds two full models at once (matters on a
# small machine — two 0.5B copies + the trainer is enough to OOM an 8 GB laptop).
tuned = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device)
_tr = SFTTrainer(model=tuned, train_dataset=sft_ds, peft_config=lora_cfg,
    args=SFTConfig(output_dir="eval_demo", dataset_text_field="text",
                   per_device_train_batch_size=2, max_steps=30, learning_rate=2e-4,
                   logging_steps=10, bf16=BF16_OK, report_to="none"))
_tr.train()
model = _tr.model.eval()           # PeftModel: adapter ON = tuned, disabled = base

# Held-out evaluation set (NOT in training) with known short answers.
eval_qa = [
    {"q": "What is the capital of Italy?", "a": "Rome"},
    {"q": "What is 9 times 6?",            "a": "54"},
    {"q": "What color do you get mixing blue and yellow?", "a": "Green"},
]
print("ready: one PeftModel (adapter on/off = tuned/base) and a held-out set")

### 1. Perplexity — the intrinsic metric

**Perplexity = exp(cross-entropy loss)** on text the model didn't train on; lower means
the model finds that text more probable. It needs no labels beyond the text itself, so
it's the first thing to check. Compute it on the *held-out* set for both models.

In [ ]:
import math

@torch.no_grad()
def perplexity(model, texts):
    losses = []
    for t in texts:
        enc = tokenizer(t, return_tensors="pt").to(model.device)
        losses.append(model(**enc, labels=enc["input_ids"]).loss.item())
    return math.exp(sum(losses) / len(losses))

held_out = [tokenizer.apply_chat_template(
    [{"role": "user", "content": x["q"]}, {"role": "assistant", "content": x["a"]}],
    tokenize=False) for x in eval_qa]

with model.disable_adapter():                       # adapter OFF -> base model
    base_ppl = perplexity(model, held_out)
tuned_ppl = perplexity(model, held_out)             # adapter ON  -> tuned model
print(f"base  perplexity: {base_ppl:6.2f}")
print(f"tuned perplexity: {tuned_ppl:6.2f}")
print("lower = better; on this tiny demo expect only a small move")

### 2. Generation diff — read the actual outputs

Numbers hide behavior. Always *look* at what changed: generate from both models on the
same prompts, greedily (deterministic), and compare side by side.

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=40):
    # return_dict=True -> a BatchEncoding we can splat into generate(); on transformers 5.x
    # apply_chat_template(return_tensors="pt") no longer returns a bare tensor.
    enc = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
            add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

for x in eval_qa:
    with model.disable_adapter():            # same weights, adapter off -> base
        base_out = generate(model, x["q"])
    tuned_out = generate(model, x["q"])      # adapter on -> tuned
    print("Q:", x["q"])
    print("  base :", base_out)
    print("  tuned:", tuned_out)
    print()

### 3. Task metric — exact-match accuracy

When examples have a known answer, score automatically. Here: does the gold answer
appear in the generation? Swap in BLEU/ROUGE (generation), F1 (classification), or
`pass@k` (code) for your task.

In [ ]:
def exact_match(model):
    hits = sum(x["a"].lower() in generate(model, x["q"]).lower() for x in eval_qa)
    return hits / len(eval_qa)

with model.disable_adapter():
    base_em = exact_match(model)
tuned_em = exact_match(model)
print(f"base  exact-match: {base_em:.0%}")
print(f"tuned exact-match: {tuned_em:.0%}")

### 4. LLM-as-judge — scoring open-ended quality

For answers with no single gold string (tone, helpfulness, style), have a **strong
model** grade each response against a rubric. In production you'd call Claude or GPT
with the prompt below and parse the score. To keep this notebook offline, `mock_judge`
is a deterministic stand-in — replace it with a real API call.

In [ ]:
JUDGE_PROMPT = """You are grading an assistant's answer.
Question: {q}
Answer: {a}
Score 1-5 for correctness and conciseness. Reply with just the number."""

def mock_judge(q, a, gold):                 # stand-in for a real strong-model call
    score = 5 if gold.lower() in a.lower() else 2     # correctness
    if len(a.split()) > 25:                           # penalize verbosity
        score -= 1
    return max(1, min(5, score))

def mean_judge_score():
    scores = [mock_judge(x["q"], generate(model, x["q"]), x["a"]) for x in eval_qa]
    return sum(scores) / len(scores)

with model.disable_adapter():
    base_judge = mean_judge_score()
tuned_judge = mean_judge_score()
print(f"base  mean judge score: {base_judge:.2f} / 5")
print(f"tuned mean judge score: {tuned_judge:.2f} / 5")
print("\nReal version: format JUDGE_PROMPT, call a strong model, parse the number.")

### 5. Standard benchmarks — `lm-evaluation-harness`

For comparable, public numbers (MMLU, ARC, HellaSwag, GSM8K, TruthfulQA, …) use EleutherAI's
harness instead of hand-rolling. Point it at the merged model (or base + adapter):

```bash
pip install lm-eval
lm_eval --model hf \
  --model_args pretrained=./my_merged_model \
  --tasks arc_easy,hellaswag,gsm8k \
  --device cuda:0 --batch_size 8
```

Report the harness version and task versions — scores drift between releases.

### Pitfalls & checklist

- **Never evaluate on training data** — it only measures memorization. Keep a real holdout.
- **Watch the split that matters:** train loss ↓ but **val loss ↑** = overfitting (notebook 01 §7).
- **Match the chat template at eval** exactly as in training, or numbers are meaningless.
- **Fix decoding** (`do_sample=False` or a set seed) so comparisons are reproducible.
- **Report versions** of model, data, and libraries with every number.

Checklist: held-out set ready · base vs. tuned compared · at least one intrinsic *and* one
task/judgment metric · decoding fixed · no train/test leakage.

In [ ]:
del model, _tr; cleanup()